In [40]:
from typing import TypedDict
from langgraph.graph import StateGraph,START,END
import re

class TextState(TypedDict):
    raw_text : str
    cleaned : str
    word_count : str
    summary : str

def clean_text(state:TextState) -> dict:
    text = state['raw_text']

    cleaned = text.strip()
    cleaned = re.sub(r'\s+', ' ', cleaned)

    cleaned = cleaned.lower()

    print(f"[clean_text] input had {len(text)} chars -> cleaned has {len(cleaned)} chars")

    return {'cleaned': cleaned}
     
    


In [41]:

def count_word(state : TextState) -> dict:
    word = state['cleaned']

    count = word.split()
    res = len(count)

    return {'word_count':res}

In [42]:
def format_result(state: TextState) -> dict:
    word_count = state['word_count']
    clean      = state['cleaned']

    preview = clean[:50] + ('...' if len(clean) > 50 else '')

    summary = (
        '=== Text Analysis Summary ===\n'
        f'Word count  : {word_count}\n'
        f'Preview     : {preview}\n'
        '=====  ========================'
    )

    print('[format_result] summary built')

    return {'summary': summary}

In [43]:
builder = StateGraph(TextState)

builder.add_node('clean_text',    clean_text)
builder.add_node('count_word',   count_word)
builder.add_node('format_result', format_result)


builder.add_edge(START,           'clean_text')     
builder.add_edge('clean_text',    'count_word')    
builder.add_edge('count_word',   'format_result')  
builder.add_edge('format_result', END)              

print('Graph edges registered.')

Graph edges registered.


In [ ]:
graph = builder.compile()

sample_text = (
    '  LangGraph is a Python library for building   stateful,\n'
    '  Multi-Step AI applications.  It uses a GRAPH structure\n'
    '  where each node is a plain Python function.  '
)

res=graph.invoke({'raw_text':sample_text})
print(res)

[clean_text] input had 161 chars -> cleaned has 150 chars
[format_result] summary built
{'raw_text': '  LangGraph is a Python library for building   stateful,\n  Multi-Step AI applications.  It uses a GRAPH structure\n  where each node is a plain Python function.  ', 'cleaned': 'langgraph is a python library for building stateful, multi-step ai applications. it uses a graph structure where each node is a plain python function.', 'word_count': 24, 'summary': '=== Text Analysis Summary ===\nWord count  : 24\nPreview     : langgraph is a python library for building statefu...\n=====  ========================'}


: 